# Processo de Escoragem

* Aplicado a base de fechamento de junho de 2025.

## 1 - Configuração e Carregamento do Modelo

In [1]:
import pandas as pd
import joblib
import numpy as np

# --- 1. SETUP DE CONFIGURAÇÃO ---
NOME_ARQUIVO_NOVO = 'base_segmentacao_jun25_fechamento.parquet' 

# Lista das 18 colunas numéricas usadas na modelagem (ordem EXATA é crucial)
COLUNAS_MODELAGEM = [
'meses_base', 'vlr_cobrado_mes_atual','vlr_pago_mes_atual','razao_pgto_vs_cobrado_mes_atual',
'tri_total_cobrado','tri_total_pago','tri_razao_pgto_vs_cobrado','dias_inadimplencia_fatura_mais_antiga',
'tri_ticket_medio_pagamento','tri_freq_pgto_atraso','tri_valor_total_em_atraso','dias_atraso',
'tri_data_max_dias_atraso','tri_qtd_bloqueio','qtd_bloqueio','razao_finalizados_vs_total_mes_matrix', 
'razao_finalizados_vs_total_trimestre_caso','razao_finalizados_vs_total_mes_workorder'
] 
# --------------------

# 2. Carregar a Nova Base de Dados (Usando pd.read_parquet)
try:
    df_mes_ref = pd.read_parquet(NOME_ARQUIVO_NOVO, engine='fastparquet')
    print(f"Base de {NOME_ARQUIVO_NOVO} carregada com {len(df_mes_ref)} observações.")
except FileNotFoundError:
    print(f"ERRO: Arquivo {NOME_ARQUIVO_NOVO} não encontrado.")
    
# 3. Carregar os Objetos PKL (Garanta que estes arquivos estejam na mesma pasta)
try:
    scaler = joblib.load('scaler_segmentacao.pkl')
    pca = joblib.load('pca_segmentacao.pkl')
    kmeans_4 = joblib.load('kmeans_4_clusters.pkl')
    print("Objetos Scaler, PCA e K-Means carregados com sucesso.")
except FileNotFoundError:
    print("ERRO: Um ou mais arquivos .pkl não foram encontrados. Interrompendo a escoragem.")
    # Adicionar uma instrução para interromper o notebook aqui, se necessário.

Base de base_segmentacao_jun25_fechamento.parquet carregada com 1013285 observações.
Objetos Scaler, PCA e K-Means carregados com sucesso.


In [2]:
#Renomear a coluna:
##Falar com Alan para ele mudar o nome de razao_pgto_vs_cobrado para razao_pgto_vs_cobrado_mes_atual
df_mes_ref.rename(columns = {'razao_pgto_vs_cobrado':'razao_pgto_vs_cobrado_mes_atual'}, inplace = True)

## 2 - Pipeline de Escoragem Automática (4 Clusters Brutos)

In [3]:
# Célula 2: Pipeline de Escoragem Automática (Versão Final e Segura)

# 1. Pré-processamento e Filtragem
print("1. Aplicando Pré-processamento (fillna(0) e filtragem de colunas)...")
df_junho = df_mes_ref.fillna(0) 
df_junho_model = df_junho[COLUNAS_MODELAGEM] 

# 2. Padronização (Scaler)
# Aplica a Média/Desvio-padrão aprendidos na base de Abril
scaled_junho = scaler.transform(df_junho_model)
print("2. Padronização (StandardScaler) concluída.")

# 3. Redução de Dimensionalidade (PCA - 6 Componentes)
# pca.transform() retorna um array NumPy de N x 6 colunas.
pca_array_junho = pca.transform(scaled_junho)

# --- PASSO CRUCIAL DE CORREÇÃO DE FEATURE NAMES ---
# 3.1. Definir a lista EXATA de 5 nomes: ['PC_1', 'PC_2', ..., 'PC_6']
# FORÇAMOS O RANGE A PARAR EM 6, como sugerido
colunas_pca_finais = [f"PC_{i+1}" for i in range(6)] 

# 3.2. CRIAR O DATAFRAME INTERMEDIÁRIO (pca_junho_final)
# Usamos o fatiamento [:, :6] para garantir que peguemos APENAS as 6 primeiras colunas (PC_1 a PC_6) 
# do array NumPy, eliminando qualquer vestígio de outros nomes.
pca_junho_final = pd.DataFrame(pca_array_junho[:, :6], columns=colunas_pca_finais)

print("3. PCA concluído.")
print(f"   Data Frame com cinco colunas no PCA: {pca_junho_final.shape}")
print(f"   Colunas enviadas ao K-Means: {pca_junho_final.columns.tolist()}")
# ----------------------------------------------------

# 4. Clusterização (K-Means)
# O kmeans_4 agora recebe um DataFrame GARANTIDO de 5 colunas e nomes corretos
labels_4_clusters = kmeans_4.predict(pca_junho_final)
df_junho['cluster_bruto'] = labels_4_clusters

print("4. Clusterização em 4 Clusters Brutos concluída com sucesso.")

1. Aplicando Pré-processamento (fillna(0) e filtragem de colunas)...
2. Padronização (StandardScaler) concluída.
3. PCA concluído.
   Data Frame com cinco colunas no PCA: (1013285, 6)
   Colunas enviadas ao K-Means: ['PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'PC_6']


C:\Users\henrique.zaidan\.conda\envs\segmentacao\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


4. Clusterização em 4 Clusters Brutos concluída com sucesso.


In [7]:
# Configurar para mostrar todas as colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Agora veja as primeiras linhas
df_junho.head(3)

,data_movimento,tipo_movimento,flg_primeiro_plano,flg_funcionario,flg_permuta,flg_migrado,flg_desconto,num,id_planos_usuarios,ativo,data_contrato,data_cadastro,meses_base,dsc_categoria,dsc_tecnologia,plano,vencimento,valor,desconto,motivo_desconto,faixa_desconto,status_plano,download,upload,vlr_scm_plano_internet,vlr_sva_plano_internet,pppoe,descricao_plano,tipo_documento,nome,documento_original,cod_ibge,dsc_municipio_ibge,cod_ibge_cto,dsc_municipio_ibge_cto,posicao_gpon,concentrador,porta,caixa,posicao,lat_cto,long_cto,login,qtd_plano_internet,vlr_plano_internet,qtd_plano_sva,vlr_plano_sva,qtd_plano_hospedagem,vlr_plano_hospedagem,qtd_plano_radio,vlr_plano_radio,qtd_plano_telefone,vlr_plano_telefone,qtd_plano_mvno,vlr_plano_mvno,qtd_plano_outros,vlr_plano_outros,data_inclusao,data_aprovisionamento,data_atualizacao,vlr_cobrado_mes_atual,vlr_pago_mes_atual,tri_total_cobrado,tri_total_pago,razao_pgto_vs_cobrado_mes_atual,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,faturado,flag_inadimplencia,status_pgto,qtd_total_mes_matrix,qtd_finalizado_mes_matrix,qtd_finalizado_inatividade_mes_matrix,razao_finalizados_vs_total_mes_matrix,qtd_total_trimestre_matrix,qtd_finalizado_trimestre_matrix,qtd_finalizado_inatividade_trimestre_matrix,razao_finalizados_vs_total_trimestre_matrix,qtd_total_mes_caso,qtd_finalizado_mes_caso,razao_finalizados_vs_total_mes_caso,qtd_total_trimestre_caso,qtd_finalizado_trimestre_caso,razao_finalizados_vs_total_trimestre_caso,qtd_total_mes_workorder,qtd_finalizado_mes_workorder,qtd_cancelado_mes_workorder,razao_finalizados_vs_total_mes_workorder,qtd_total_trimestre_workorder,qtd_finalizado_trimestre_workorder,qtd_cancelado_trimestre_workorder,razao_finalizados_vs_total_trimestre_workorder,desconto_bloqueio_mes,tri_desconto_bloqueio_mes,flg_negociacao,tri_flg_negociacao,qtd_parcelas,parcela_atual,vlr_base,vlr_devido,qtd_bloqueio,tri_qtd_bloqueio,tipo_churn,data_exclusao,cluster_bruto
0,2025-06-01,base_final,1,0,0,0,0,6,2668640,385867,2014-01-01,2014-01-01,138,Fibra Home Combo,Fibra,T-G-25.600M.6000G,10,109.99,0.0,0,Cliente sem desconto,A,600000,300000,109.99,0.0,Turbo-G-600M,Fibra Home Combo 600M+ 2025,PF,Pedro Piazentin Neto,135.347.598-00,3552403,Sumaré,3552403,Sumaré,2420261,468,42008,13,105,-22.829052,-47.268673,6.1@desktop.com.br,1,109.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2025-01-24 08:10:50,2025-01-24 08:14:45,2025-10-24,109.99,109.99,329.97,329.97,1.0,1.0,0,109.99,0.0,0.0,-5.0,-5.0,1,0.0,antecipado,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0
1,2025-06-01,base_final,1,0,0,0,0,9,3147322,2091936,2006-03-03,2006-03-03,231,Fibra Home Combo,Fibra,T-G-24.400M.4000G,15,99.99,0.0,0,Cliente sem desconto,A,400000,200000,99.99,0.0,Turbo-G-400M,Fibra Home Combo 400M 2024 *,PF,Fabricio de Lima Moraes,121.668.688-27,3552403,Sumaré,3552403,Sumaré,2954960,558,52781,2484,18690,-22.828738,-47.211647,9.2@desktop.com.br,1,99.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2025-06-06 18:55:35,2025-06-06 18:52:58,2025-10-24,84.99,84.99,254.97,254.97,1.0,1.0,0,84.99,1.0,0.0,1.0,1.0,1,0.0,em_dia,1.0,1.0,0.0,1.0,2.0,1.0,1.0,1.0,4.0,4.0,1.0,10.0,10.0,1.0,1.0,1.0,0.0,1.0,3.0,1.0,0.0,0.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,2
2,2025-06-01,base_final,1,0,0,0,0,12,2338539,1925310,2025-06-01,2005-03-18,243,Fibra Home Combo,Fibra,T-G-25.700M.4000G,20,119.99,0.0,0,Cliente sem desconto,A,700000,350000,119.99,0.0,Turbo-G-1G,Fibra Home Combo 700M 2025 *,PF,Rui Savitsky,619.721.268-49,3552403,Sumaré,3552403,Sumaré,281345,227,5175,340,2713,-22.799621,-47.216536,12.1@desktop.com.br,1,119.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,2024-09-25 16:09:32,2025-10-24,114.99,114.99,344.97,344.97,1.0,1.0,0,114.99,0.0,0.0,-12.0,-10.0,1,0.0,antecipado,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.

## 3 - Lógica de Negócio (Fusão e Segmentação Final)

In [10]:
# Célula 3: Aplicação da Lógica de Negócio (Fusão e Sub-Segmentação)

# 1. Lógica da Fusão (4 -> 3 Clusters Estatísticos)
# O Cluster 3 (bruto) é reatribuído ao Cluster 2
df_junho['cluster_final'] = df_junho['cluster_bruto']
df_junho.loc[df_junho['cluster_final'] == 3, 'cluster_final'] = 2 
print("-> Fusão Clusters 3 e 2 concluída (3 Clusters Estatísticos restantes: 0, 1, 2).")

# 2. Lógica de Segmentação Final (Criação da Variável de Negócio)
# O Cluster 0 (Risco Baixo) é dividido no grupo Premium e no restante Risco Baixo
conditions = [
    df_junho['cluster_final'] == 1, # Alto Risco (Cluster 1)
    df_junho['cluster_final'] == 2, # Risco Médio (Cluster 2+3 fundidos)
    
    # Cliente Premium: Clientes no Cluster 0 E (Longevidade >= 26) E (Tri Pago >= 319) E (Atraso <= 0)
    (df_junho['cluster_final'] == 0) & (df_junho['meses_base'] >= 26) & 
    (df_junho['tri_total_pago'] >= 319) & (df_junho['tri_data_max_dias_atraso'] <= 0)
]

choices = [
    'alto_risco',
    'risco_medio',
    'cliente_premium'
]

# O valor default (restante do Cluster 0) é 'risco_baixo'
df_junho['segmentacao'] = np.select(conditions, choices, default='risco_baixo') 
print("-> Variável 'segmentacao' final criada.")

-> Fusão Clusters 3 e 2 concluída (3 Clusters Estatísticos restantes: 0, 1, 2).
-> Variável 'segmentacao' final criada.


## 4 - Tabela Final 

In [11]:
df_junho.head(3)

,data_movimento,tipo_movimento,flg_primeiro_plano,flg_funcionario,flg_permuta,flg_migrado,flg_desconto,num,id_planos_usuarios,ativo,data_contrato,data_cadastro,meses_base,dsc_categoria,dsc_tecnologia,plano,vencimento,valor,desconto,motivo_desconto,faixa_desconto,status_plano,download,upload,vlr_scm_plano_internet,vlr_sva_plano_internet,pppoe,descricao_plano,tipo_documento,nome,documento_original,cod_ibge,dsc_municipio_ibge,cod_ibge_cto,dsc_municipio_ibge_cto,posicao_gpon,concentrador,porta,caixa,posicao,lat_cto,long_cto,login,qtd_plano_internet,vlr_plano_internet,qtd_plano_sva,vlr_plano_sva,qtd_plano_hospedagem,vlr_plano_hospedagem,qtd_plano_radio,vlr_plano_radio,qtd_plano_telefone,vlr_plano_telefone,qtd_plano_mvno,vlr_plano_mvno,qtd_plano_outros,vlr_plano_outros,data_inclusao,data_aprovisionamento,data_atualizacao,vlr_cobrado_mes_atual,vlr_pago_mes_atual,tri_total_cobrado,tri_total_pago,razao_pgto_vs_cobrado_mes_atual,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,faturado,flag_inadimplencia,status_pgto,qtd_total_mes_matrix,qtd_finalizado_mes_matrix,qtd_finalizado_inatividade_mes_matrix,razao_finalizados_vs_total_mes_matrix,qtd_total_trimestre_matrix,qtd_finalizado_trimestre_matrix,qtd_finalizado_inatividade_trimestre_matrix,razao_finalizados_vs_total_trimestre_matrix,qtd_total_mes_caso,qtd_finalizado_mes_caso,razao_finalizados_vs_total_mes_caso,qtd_total_trimestre_caso,qtd_finalizado_trimestre_caso,razao_finalizados_vs_total_trimestre_caso,qtd_total_mes_workorder,qtd_finalizado_mes_workorder,qtd_cancelado_mes_workorder,razao_finalizados_vs_total_mes_workorder,qtd_total_trimestre_workorder,qtd_finalizado_trimestre_workorder,qtd_cancelado_trimestre_workorder,razao_finalizados_vs_total_trimestre_workorder,desconto_bloqueio_mes,tri_desconto_bloqueio_mes,flg_negociacao,tri_flg_negociacao,qtd_parcelas,parcela_atual,vlr_base,vlr_devido,qtd_bloqueio,tri_qtd_bloqueio,tipo_churn,data_exclusao,cluster_bruto,cluster_final,segmentacao
0,2025-06-01,base_final,1,0,0,0,0,6,2668640,385867,2014-01-01,2014-01-01,138,Fibra Home Combo,Fibra,T-G-25.600M.6000G,10,109.99,0.0,0,Cliente sem desconto,A,600000,300000,109.99,0.0,Turbo-G-600M,Fibra Home Combo 600M+ 2025,PF,Pedro Piazentin Neto,135.347.598-00,3552403,Sumaré,3552403,Sumaré,2420261,468,42008,13,105,-22.829052,-47.268673,6.1@desktop.com.br,1,109.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2025-01-24 08:10:50,2025-01-24 08:14:45,2025-10-24,109.99,109.99,329.97,329.97,1.0,1.0,0,109.99,0.0,0.0,-5.0,-5.0,1,0.0,antecipado,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,cliente_premium
1,2025-06-01,base_final,1,0,0,0,0,9,3147322,2091936,2006-03-03,2006-03-03,231,Fibra Home Combo,Fibra,T-G-24.400M.4000G,15,99.99,0.0,0,Cliente sem desconto,A,400000,200000,99.99,0.0,Turbo-G-400M,Fibra Home Combo 400M 2024 *,PF,Fabricio de Lima Moraes,121.668.688-27,3552403,Sumaré,3552403,Sumaré,2954960,558,52781,2484,18690,-22.828738,-47.211647,9.2@desktop.com.br,1,99.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2025-06-06 18:55:35,2025-06-06 18:52:58,2025-10-24,84.99,84.99,254.97,254.97,1.0,1.0,0,84.99,1.0,0.0,1.0,1.0,1,0.0,em_dia,1.0,1.0,0.0,1.0,2.0,1.0,1.0,1.0,4.0,4.0,1.0,10.0,10.0,1.0,1.0,1.0,0.0,1.0,3.0,1.0,0.0,0.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,2,2,risco_medio
2,2025-06-01,base_final,1,0,0,0,0,12,2338539,1925310,2025-06-01,2005-03-18,243,Fibra Home Combo,Fibra,T-G-25.700M.4000G,20,119.99,0.0,0,Cliente sem desconto,A,700000,350000,119.99,0.0,Turbo-G-1G,Fibra Home Combo 700M 2025 *,PF,Rui Savitsky,619.721.268-49,3552403,Sumaré,3552403,Sumaré,281345,227,5175,340,2713,-22.799621,-47.216536,12.1@desktop.com.br,1,119.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,2024-09-25 16:09:32,2025-10-24,114.99,114.99,344.97,344.97,1.0,1.0,0,114.99,0.0,0.0,-12.0,-10.0,1,0.0,antecipado,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [13]:
tabela_final_junho = df_junho[['num', 'data_movimento', 'segmentacao', 'tipo_churn',  'data_exclusao']]

# Aplicação do reset_index(drop=True)
tabela_final_junho = tabela_final_junho.reset_index(drop=True)

tabela_final_junho.head(3)

,num,data_movimento,segmentacao,tipo_churn,data_exclusao
0,6,2025-06-01,cliente_premium,0,0
1,9,2025-06-01,risco_medio,0,0
2,12,2025-06-01,cliente_premium,0,0


In [19]:
# Primeiro converter para datetime
tabela_final_junho['data_movimento'] = pd.to_datetime(tabela_final_junho['data_movimento'])

# Agora converter para o formato desejado (string no formato data)
tabela_final_junho['data_movimento'] = tabela_final_junho['data_movimento'].dt.strftime('%Y-%m-%d')

# Converter a coluna tipo_churn para string também
tabela_final_junho['tipo_churn'] = tabela_final_junho['tipo_churn'].astype(str)

In [21]:
tabela_final_junho.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1013285 entries, 0 to 1013284
Data columns (total 5 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   num             1013285 non-null  int64 
 1   data_movimento  1013285 non-null  object
 2   segmentacao     1013285 non-null  object
 3   tipo_churn      1013285 non-null  object
 4   data_exclusao   1013285 non-null  object
dtypes: int64(1), object(4)
memory usage: 38.7+ MB


In [23]:
#Forço que as colunas que são object se tornem string
for coluna in tabela_final_junho.select_dtypes(include=['object']).columns:
    tabela_final_junho[coluna] = tabela_final_junho[coluna].astype(str)

In [24]:
##Salvar em parquet
tabela_final_junho.to_parquet('base_junho_segmentacao_escorada.parquet', engine='pyarrow')